# Day 10 | ILT 3: Databricks Workflows vs Airflow — Deep Dive, Task Dependencies, Retries & Failure Handling
### GlobalMart Data Engineering Bootcamp
---
**Duration:** 90 minutes &nbsp;|&nbsp; **Level:** Intermediate &nbsp;|&nbsp; **Tags:** orchestration, databricks-workflows, airflow, task-dependencies, retries, failure-handling

---
**Builds on:** the earlier ILT this afternoon — *Introduction to Orchestration: Need, DAG Concepts & Workflow Design*. This session assumes you already know what a DAG is (nodes = tasks, edges = dependencies, no cycles) and why GlobalMart's Bronze → Silver → Gold pipeline needs one. Today goes one level deeper: **which** orchestrator, and **exactly how** dependencies, retries, and failure handling work under the hood.

---
**Goal:** Compare Databricks Workflows and Apache Airflow the way an engineer actually would when choosing between them — then implement, in real runnable Python (no Databricks API calls anywhere), the three mechanics every orchestrator has to solve: task dependencies, retries, and failure handling.

---
**INSTRUCTOR NOTE:**
This is a code-demo ILT, not a pure lecture — every code cell below actually runs and prints real output. But nothing in this notebook creates, starts, schedules, or triggers a real Databricks Job, Workflow, Pipeline, SQL Warehouse, or cluster — there is no `dbutils.jobs`, no `WorkspaceClient()` job/pipeline call, and no Jobs/Pipelines REST call anywhere in this file. Building and running the real Workflow happens hands-on on Day 11. Today is entirely about the mechanics and vocabulary, demonstrated safely in plain Python.

## Learning Objectives

By the end of this session, students will be able to:

1. Compare Databricks Workflows and Apache Airflow across task types, cluster management, authoring, UI/observability, and cost model
2. State a concrete decision rule for choosing one over the other — a Databricks-only shop vs. an org already standardized on Airflow for cross-system orchestration
3. Explain `depends_on`, fan-out/fan-in, and what happens by default when an upstream task fails
4. Explain Databricks Jobs' retry fields (`max_retries`, `min_retry_interval_millis`, `retry_on_timeout`) and contrast them with Airflow's `retries`/`retry_delay`
5. Explain `run_if` conditions and design a dedicated on-failure/alert task — and contrast with Airflow's `trigger_rule`
6. Trace a task failure through a dependency graph to predict exactly which downstream tasks run, skip, or fire *because of* the failure

---
## Section 1: Databricks Workflows vs. Apache Airflow

Both are orchestrators — something has to call Bronze, then Silver, then Gold, in the right order, on a schedule, and tell you when it breaks. The question isn't "which one is better," it's "which one fits the org actually doing the orchestrating."

| | Databricks Workflows | Apache Airflow |
|---|---|---|
| **Scope** | Native to Databricks — built specifically to orchestrate Databricks-native work | General-purpose — orchestrates *anything* an operator exists for: Databricks, Snowflake, S3, dbt, Kubernetes, on-prem systems, plain APIs |
| **Task types** | Notebook, Python script, SQL, dbt, Lakeflow/DLT pipeline, JAR, "Run Job" — all first-class, built into the UI, zero extra install | Whatever a provider package supplies — `DatabricksSubmitRunOperator`/`DatabricksRunNowOperator`, `PostgresOperator`, `S3Operator`, `PythonOperator`... 90+ provider packages. Databricks awareness is an add-on, not native |
| **Cluster/compute management** | Spins up a **job cluster automatically per run**, sized per task, and terminates it when the run ends — you declare the compute, Workflows owns the lifecycle | Doesn't manage compute at all. A `DatabricksSubmitRunOperator` calls out to the Databricks Jobs REST API over a connection you configure; Databricks still starts/stops the cluster — Airflow just fires the remote call and polls for status |
| **Authoring** | Point-and-click UI, JSON, Databricks Asset Bundles (YAML), or Terraform | Python DAG files — "configuration as code," everything is a `DAG()` object in a `.py` file |
| **Scheduling engine** | Built into the Databricks control plane — nothing extra to run | A separate service you operate yourself: scheduler, webserver, metadata database, workers — real infrastructure you patch and scale |
| **UI / observability** | Per-task Gantt-style run view inside the Databricks workspace — one click from a failed task straight into its Spark UI / driver logs | DAG graph and grid views in the Airflow webserver — solid for task status and logs, but reaching a Spark UI or cluster metric means leaving Airflow entirely |
| **Governance/lineage** | Unity Catalog lineage is automatic for every Databricks task — no extra wiring | No native Unity Catalog awareness — lineage across an Airflow DAG has to be stitched together separately |
| **Cost model** | Job-compute pricing (cheaper than all-purpose clusters), billed only while a run is actually executing | Airflow itself needs infrastructure (VM/Kubernetes + metadata DB) running **continuously**, independent of whatever compute the tasks it triggers cost |

### When you'd actually choose one over the other

- **A Databricks-only shop — like GlobalMart today.** Every source lands in `gbmart` and every transformation is a Databricks notebook or pipeline. Workflows wins: no separate infrastructure to run, faster to author, and Unity Catalog lineage comes for free.
- **An org already standardized on Airflow for cross-system orchestration.** If Databricks is one stop among many — a nightly DAG that also refreshes a Snowflake mart, calls a Salesforce API, and drops a file over SFTP to an on-prem system — Airflow wins, because it's the one tool that can express the *whole* company DAG, not just the Databricks slice of it.
- **The common hybrid in practice:** Airflow as the company-wide orchestrator, where one Airflow task's entire job is "kick off this one Databricks Workflow and wait for it to finish" (via `DatabricksRunNowOperator`). Each tool does the part it's actually good at — Airflow coordinates across systems, Workflows coordinates *inside* Databricks.

**Grounding this in GlobalMart's real sources — and it's 2, not 4.** Day 3 ILT 1 and Day 4 ILT 1 already established this explicitly: GlobalMart's production pipeline has exactly **two** real ingestion pathways — Postgres/Supabase CDC via Lakeflow Connect (`orders_data_ingestion_cdc`, landing `orders`/`order_items`) and ADLS Autoloader (landing `customers`/`products`/`address`/`payments`/`payment_methods`). The REST API and Neo4j/GraphDB patterns from Day 3 are deliberate side-explorations that land in a `sandbox/` path — they never touch a real `gbmart.bronze` table, so they don't get a `task_key` in any real GlobalMart job, here or on Day 11. Both real pathways feed one Databricks-only pipeline today, so Workflows is the right default, and exactly what Day 11's hands-on builds. If GlobalMart's Postgres source were ever owned by a different team running its own Airflow instance for reasons that have nothing to do with Databricks, *that's* the moment a cross-tool Airflow DAG — with GlobalMart's entire Bronze→Silver→Gold Workflow as a single task inside it — would start to make sense.</cell id="cell-workflows-vs-airflow-03">


In [ ]:
# --- The running example for the rest of this session --------------------
# A small, GlobalMart-shaped Databricks Workflow using GlobalMart's REAL
# ingestion architecture: exactly 2 pathways (Day 3 ILT 1 / Day 4 ILT 1 /
# this afternoon's earlier ILT2 all establish this) -- Postgres/Supabase CDC
# and ADLS Autoloader. They fan OUT in parallel, converge (fan IN) into
# build_silver, then build_gold_fact_sales, plus one dedicated
# on_failure_alert task (Section 4). This mirrors the real "tasks" array
# shape inside a Databricks Jobs API create/reset payload -- task_key,
# depends_on, and the retry fields from Section 3 -- but it is a plain
# Python list of dicts. Nothing here calls the Jobs API, dbutils.jobs, or
# WorkspaceClient() -- it never creates, starts, or schedules a real job.
#
# NOTE: this is a coarser grain than the earlier ILT2's task graph, which
# modeled all 7 real Bronze tables as 7 separate tasks. ILT2 itself called
# this out as "a design choice, not a law" -- today's session is about
# retries and failure handling, not re-teaching the full DAG, so the 2 real
# pathways are grouped into one task each on purpose.

import json

databricks_workflow_tasks = [
    {   # fan-out: both tasks below have no depends_on, so they start together
        "task_key": "ingest_bronze_cdc",               # Postgres/Supabase CDC -> gbmart.bronze.orders / order_items
        "depends_on": [],
        "max_retries": 2,
        "min_retry_interval_millis": 60000,
        "retry_on_timeout": True,
    },
    {
        "task_key": "ingest_bronze_autoloader",        # ADLS Autoloader -> gbmart.bronze.customers/products/address/payments/payment_methods
        "depends_on": [],
        "max_retries": 1,
        "min_retry_interval_millis": 30000,
        "retry_on_timeout": False,
    },
    {   # fan-in: waits for BOTH real ingestion tasks above (default run_if = ALL_SUCCESS)
        "task_key": "build_silver",
        "depends_on": [
            {"task_key": "ingest_bronze_cdc"},
            {"task_key": "ingest_bronze_autoloader"},
        ],
        "max_retries": 1,
        "min_retry_interval_millis": 120000,
        "retry_on_timeout": True,
    },
    {
        "task_key": "build_gold_fact_sales",
        "depends_on": [{"task_key": "build_silver"}],
        "max_retries": 1,
        "min_retry_interval_millis": 120000,
        "retry_on_timeout": True,
    },
    {   # dedicated on-failure task (Section 4) -- note the run_if override
        "task_key": "on_failure_alert",
        "depends_on": [
            {"task_key": "ingest_bronze_cdc"},
            {"task_key": "ingest_bronze_autoloader"},
            {"task_key": "build_silver"},
            {"task_key": "build_gold_fact_sales"},
        ],
        "max_retries": 0,
        "min_retry_interval_millis": 0,
        "retry_on_timeout": False,
        "run_if": "AT_LEAST_ONE_FAILED",   # <- opposite of every other task's implicit ALL_SUCCESS
    },
]

print(f"Databricks Workflow -- {len(databricks_workflow_tasks)} tasks\n")
for t in databricks_workflow_tasks:
    deps = ", ".join(d["task_key"] for d in t["depends_on"]) or "(none -- start of DAG)"
    print(f"  {t['task_key']:<26} depends_on: {deps}")

print("\nFull shape of one task, e.g. the fan-in task 'build_silver':\n")
print(json.dumps(databricks_workflow_tasks[2], indent=2))


In [ ]:
# --- The same pipeline, Airflow-style -- TEXT ONLY, never executed --------
# Airflow is not installed in this environment, and this variable is a plain
# multi-line string -- it is never imported, parsed, or run. It exists purely
# so you can visually compare Airflow's syntax to the Databricks dict above.

airflow_equivalent_dag = '''
from airflow import DAG
from airflow.providers.databricks.operators.databricks import DatabricksSubmitRunOperator
from datetime import datetime, timedelta

default_args = {
    "retries": 2,
    "retry_delay": timedelta(minutes=1),
}

with DAG(
    "globalmart_bronze_to_gold",
    start_date=datetime(2026, 1, 1),
    schedule_interval="@daily",
    default_args=default_args,
) as dag:

    ingest_cdc        = DatabricksSubmitRunOperator(task_id="ingest_bronze_cdc")
    ingest_autoloader = DatabricksSubmitRunOperator(task_id="ingest_bronze_autoloader", retries=1)

    build_silver          = DatabricksSubmitRunOperator(task_id="build_silver", retries=1)
    build_gold_fact_sales = DatabricksSubmitRunOperator(task_id="build_gold_fact_sales", retries=1)

    on_failure_alert = DatabricksSubmitRunOperator(
        task_id="on_failure_alert",
        trigger_rule="one_failed",   # Airflow's closest equivalent to run_if=AT_LEAST_ONE_FAILED
    )

    # fan-out (GlobalMart's 2 real ingestion pathways), then fan-in with the ">>" operator
    [ingest_cdc, ingest_autoloader] >> build_silver >> build_gold_fact_sales
    [ingest_cdc, ingest_autoloader, build_silver, build_gold_fact_sales] >> on_failure_alert
'''

print(airflow_equivalent_dag)


---
## Section 2: Task Dependencies — `depends_on`, Fan-Out/Fan-In

**INSTRUCTOR NOTE:**
The earlier ILT this afternoon already established *why* a DAG has no cycles and *why* dependencies matter. This section is the syntax: how Databricks Workflows actually expresses "wait for this" in the Jobs API, using the exact task list from the cell above.

---

Look back at `databricks_workflow_tasks`: every task has a `depends_on` field — a list of `{"task_key": ...}` objects.

| `depends_on` value | Meaning |
|---|---|
| `[]` (empty) | No upstream — this task can start as soon as the run begins |
| `[{"task_key": "x"}]` | Simple chain — wait for exactly one upstream task |
| `[{"task_key": "a"}, {"task_key": "b"}, ...]` | **Fan-in** — wait for *every* listed task before starting |

### Fan-out

`ingest_bronze_cdc` and `ingest_bronze_autoloader` both have `depends_on: []` — neither waits on the other, so Databricks Workflows runs both **concurrently**, each on its own job cluster. This is GlobalMart's real shape: two independent pathways, no reason to force them into a single-file queue.

### Fan-in

`build_silver` lists both real ingestion tasks in its `depends_on` — it will not start until **both** of them have finished successfully. This exact shape — parallel ingestion tasks fanning into one Silver build — is precisely the pattern Day 11's hands-on lab has you build for real: *"Build Databricks Workflow: 4 Parallel Ingestion Tasks to Bronze-Silver-Gold."* (That calendar title's "4" refers to 4 parallel Bronze-table-level tasks, not 4 source systems — the earlier ILT2 today modeled the full 7-table breakdown; this session's `databricks_workflow_tasks` groups the same 2 real pathways more coarsely, on purpose, to keep the retry/failure-handling demo focused.)

### What happens when an upstream task fails?

By default, every task's implicit condition is `run_if = ALL_SUCCESS` (Section 4 covers this field properly). That means: if **any** task in `depends_on` did not succeed, the downstream task is **not run at all** — Databricks marks it **SKIPPED**, not failed, not run-with-partial-data. This cascades: if `ingest_bronze_autoloader` fails, `build_silver` is skipped, and since `build_gold_fact_sales` depends on `build_silver`, it's skipped too — one failure quietly skips the rest of the chain, unless you deliberately design around it. That's exactly what Section 4's `on_failure_alert` task exists to catch.</cell id="cell-task-dependencies-06">


---
## Section 3: Retries

Every task in `databricks_workflow_tasks` also carries three retry fields — real Databricks Jobs API fields, not simplified stand-ins:

| Field | Type | Meaning |
|---|---|---|
| `max_retries` | int | How many additional attempts after the first failed one (`max_retries: 2` = up to 3 total attempts) |
| `min_retry_interval_millis` | int | The **minimum** wait, in milliseconds, before the next retry attempt starts |
| `retry_on_timeout` | bool | Whether a task that times out (as opposed to erroring) is also eligible for retry |

### Contrast with Airflow

| | Databricks Jobs (per task) | Airflow (per task/operator) |
|---|---|---|
| Retry count | `max_retries` (int) | `retries` (int) |
| Delay between attempts | `min_retry_interval_millis` — a flat minimum, set once | `retry_delay` (a `timedelta`); Airflow can also grow the delay itself with `retry_exponential_backoff=True` + a `max_retry_delay` cap |
| Timeout handling | `retry_on_timeout` (bool) — an explicit switch | Governed by `execution_timeout` combined with the same `retries` machinery — no separate timeout-specific flag |
| Where it's set | Per task, inside that task's block in the Jobs API/UI | Per operator, or once for the whole DAG via `default_args` (as seen in the Airflow snippet above) |

**Databricks' `min_retry_interval_millis` does not grow on its own** — it is the same minimum wait every time, for that task. If you want a truly increasing (exponential) delay on Databricks, you implement it yourself inside the task's own code — which is exactly what the simulation below does, in plain Python, so you can see the mechanic instead of just reading about it.

In [ ]:
# --- Retry-with-backoff simulation -- real, executable Python -------------
# This reimplements the *mechanic* behind max_retries / min_retry_interval_millis
# (and Airflow's retries / retry_delay) using nothing but random, time.sleep,
# and a plain function -- no Databricks API, no dbutils, no network call.
# Sleep values are kept tiny (well under a second per attempt) so this cell
# finishes almost instantly under "Run All".

import random
import time

random.seed(42)   # fixed seed -> identical, reproducible output every time this runs

def simulate_flaky_call(task_key, attempt, fail_until):
    """Pretends to call one of GlobalMart's ingestion tasks. Fails on every
    attempt up to and including fail_until, then succeeds. No network
    call happens -- this only ever raises/returns based on plain numbers."""
    if attempt <= fail_until:
        raise ConnectionError(f"{task_key}: simulated transient failure on attempt {attempt}")
    return f"{task_key}: succeeded on attempt {attempt}"


def run_with_retry(task_key, max_retries, base_delay=0.4, force_exhaust=False):
    """The retry loop itself -- up to max_retries + 1 total attempts, with an
    INCREASING delay between them (base_delay * attempt), mirroring the idea
    behind min_retry_interval_millis. force_exhaust is only used once below,
    to deterministically show what happens when every retry is used up."""
    fail_until = (max_retries + 1) if force_exhaust else random.randint(1, max_retries + 1)
    for attempt in range(1, max_retries + 2):
        try:
            result = simulate_flaky_call(task_key, attempt, fail_until)
            print(f"  [{task_key}] attempt {attempt}/{max_retries + 1}: SUCCESS -> {result}")
            return "SUCCEEDED"
        except ConnectionError as e:
            if attempt == max_retries + 1:
                print(f"  [{task_key}] attempt {attempt}/{max_retries + 1}: FAILED -> {e} (max_retries exhausted, giving up)")
                return "FAILED"
            delay = round(base_delay * attempt, 2)
            print(f"  [{task_key}] attempt {attempt}/{max_retries + 1}: failed -> {e}  (retrying in {delay}s)")
            time.sleep(delay)   # small, increasing delay -- seconds, never minutes


retry_outcomes = {}

print("=== ingest_bronze_cdc  (max_retries=2) ===")
retry_outcomes["ingest_bronze_cdc"] = run_with_retry("ingest_bronze_cdc", max_retries=2)

print("\n=== ingest_bronze_autoloader  (max_retries=1, deliberately exhausted) ===")
retry_outcomes["ingest_bronze_autoloader"] = run_with_retry("ingest_bronze_autoloader", max_retries=1, force_exhaust=True)

print("\nFinal outcome per task, after retries:", retry_outcomes)


---
## Section 4: Failure Handling

`ingest_bronze_autoloader` just exhausted its retries and FAILED for real, in the cell above. This section is about what a well-designed Workflow does *because* of that.

### Task-level failure notifications

Databricks Jobs supports `email_notifications` and `webhook_notifications`, configurable per task or per job — `on_start` / `on_success` / `on_failure` — fired automatically the moment a task's run finishes, no polling required.

### `run_if` — conditional execution based on upstream outcome

| `run_if` value | Runs when... |
|---|---|
| `ALL_SUCCESS` (default) | Every task in `depends_on` succeeded |
| `AT_LEAST_ONE_SUCCESS` | At least one upstream task succeeded |
| `NONE_FAILED` | No upstream task failed (skips still allowed) |
| `ALL_DONE` | Every upstream task finished, regardless of outcome |
| `AT_LEAST_ONE_FAILED` | At least one upstream task failed |
| `ALL_FAILED` | Every upstream task failed |

### The dedicated on-failure task pattern

Look back at `on_failure_alert` in `databricks_workflow_tasks`: it lists **every** task as a dependency, and sets `"run_if": "AT_LEAST_ONE_FAILED"`. That inverts the default logic — this task is specifically designed to stay `SKIPPED` when everything goes fine, and to fire (send the Slack/email/PagerDuty alert, kick off a cleanup job) exactly when something didn't.

### Contrast with Airflow

| | Databricks Jobs | Airflow |
|---|---|---|
| Conditional run based on upstream outcome | `run_if` on the task | `trigger_rule` on the task (`all_success` default, `one_failed`, `all_done`, ...) |
| Automatic notification on failure | `email_notifications` / `webhook_notifications` block | `on_failure_callback` (a Python function), or a dedicated Slack/PagerDuty/email operator |
| Dedicated "if anything failed, do X" task | A task with `depends_on` on everything it watches + `run_if: AT_LEAST_ONE_FAILED` | A task with `trigger_rule="one_failed"` downstream of everything it watches |

The simulation below walks the exact `databricks_workflow_tasks` graph, using the real outcome from Section 3 (`ingest_bronze_autoloader` = FAILED), and predicts every other task's status.</cell id="cell-failure-handling-09">


In [ ]:
# --- Failure-propagation simulation -- real, executable Python ------------
# Walks the SAME databricks_workflow_tasks graph from earlier, starting from
# whichever tasks actually ended up FAILED in the retry simulation above, and
# works out every other task's status by following depends_on. Plain dict/
# graph traversal only -- no external calls, nothing Databricks-specific.

def propagate_failure(tasks, failed_task_keys):
    """
    tasks must already be listed in topological order (each task's
    dependencies appear earlier in the list) -- true for
    databricks_workflow_tasks as defined above.

    Returns one of "FAILED", "SUCCESS", "SKIPPED", or
    "SUCCESS (ran on failure)" for every task_key.
    """
    status = {}
    for t in tasks:
        key = t["task_key"]
        deps = [d["task_key"] for d in t["depends_on"]]
        run_if = t.get("run_if", "ALL_SUCCESS")   # Databricks' own default when run_if is omitted

        if key in failed_task_keys:
            status[key] = "FAILED"
        elif run_if == "AT_LEAST_ONE_FAILED":
            status[key] = "SUCCESS (ran on failure)" if any(status.get(d) == "FAILED" for d in deps) else "SKIPPED"
        elif any(status.get(d) in ("FAILED", "SKIPPED") for d in deps):
            status[key] = "SKIPPED"
        else:
            status[key] = "SUCCESS"
    return status


# Reuse Section 3's real outcome instead of hardcoding it a second time
failed_task_keys = {task_key for task_key, outcome in retry_outcomes.items() if outcome == "FAILED"}
print("Tasks that ultimately failed after exhausting retries:", failed_task_keys or "(none)")

final_status = propagate_failure(databricks_workflow_tasks, failed_task_keys)

print(f"\n{'TASK':<28}{'FINAL STATUS'}")
print("-" * 50)
for t in databricks_workflow_tasks:
    print(f"{t['task_key']:<28}{final_status[t['task_key']]}")


---
## Session Summary

| Topic | Databricks Workflows | Apache Airflow |
|---|---|---|
| Best fit | Databricks-only pipelines | Cross-system orchestration, Databricks as one node among many |
| Cluster management | Automatic, per-run job clusters | None — delegates to Databricks via an operator/connection |
| Task dependency syntax | `depends_on: [{"task_key": ...}]` | `upstream >> downstream` |
| Retry fields | `max_retries`, `min_retry_interval_millis`, `retry_on_timeout` | `retries`, `retry_delay` (+ optional exponential backoff) |
| Conditional execution | `run_if` | `trigger_rule` |

### Key Takeaways

- Databricks Workflows and Airflow solve the same problem for different scopes — native-and-simple vs. general-and-everywhere. GlobalMart is Databricks-only today, so Workflows is the right default; that changes the moment orchestration needs to reach outside Databricks.
- `depends_on` is how fan-out and fan-in are expressed — GlobalMart's 2 real ingestion pathways (CDC, Autoloader) fan out from the start of the run, then fan back in at `build_silver`.
- A task with an unsatisfied `depends_on` isn't retried and isn't failed — it's **SKIPPED**, and that skip cascades to everything downstream of it, by default.
- `max_retries` / `min_retry_interval_millis` / `retry_on_timeout` give a task its own retry policy; Databricks' interval is a flat minimum, not automatic exponential backoff the way Airflow's `retry_exponential_backoff` can be.
- `run_if: AT_LEAST_ONE_FAILED` is what turns a normal task into a dedicated on-failure/alert task — the one place in this whole DAG that's *supposed* to fire because something else broke.
- Every simulation in this notebook ran in plain Python — the real Workflow gets built for real next.

---

### What's Next

Day 11's first hands-on session builds a real Databricks Job around exactly this pattern: **"Build Databricks Workflow — 4 Parallel Ingestion Tasks to Bronze-Silver-Gold."** That title's "4" is at the Bronze-table level (matching the earlier ILT2's 7-table breakdown, of which 4 are the largest parallel batch), not 4 source systems — GlobalMart still has exactly 2 real ingestion pathways, CDC and Autoloader. The `databricks_workflow_tasks` list above stops being a Python dict and becomes an actual Databricks Job — same real fan-out/fan-in shape, same retries and failure handling, now wired to real notebook tasks against `gbmart`.</cell id="cell-summary-11">
